# 04 - Explicabilidad Forense en Isolation Forest: Atribución de Variables y Reglas de Fraude

## Curso de Machine Learning: Modelos No Supervisados

En la industria bancaria y aseguradora, bloquear una tarjeta de crédito o congelar una transferencia empresarial tiene consecuencias legales y comerciales directas. Si un modelo `IsolationForest` arroja `score = -0.68 (FRAUDE)`, el oficial de cumplimiento exigirá saber de inmediato: **"¿Cuáles de las 20 variables de esta transacción causaron que el bosque la aislara tan rápido?"**

A diferencia de un árbol supervisado estándar, `IsolationForest` en Scikit-Learn **no tiene un atributo `.feature_importances_` global**, porque las divisiones en cada árbol se eligen al azar.

En este cuaderno implementaremos tres técnicas de **Explicabilidad Forense (XAI) para Isolation Forest**:
1. **Importancia Global por Permutación sobre el Puntaje de Aislamiento**.
2. **Atribución Local por Transacción Individual (*Feature-level Isolation Contribution*)**: Midiendo cómo cambia el puntaje de anomalía al reemplazar cada variable sospechosa por la mediana normal.
3. **Destilación en un Árbol de Decisión Sustituto (*Surrogate Rule Extractor*)** para traducir el bosque en reglas explícitas de motor antifraude.

---

### Contenidos
1. El Reto de la Caja Negra en Ensambles de Aislamiento Aleatorio
2. Importancia Global de Variables: Sensibilidad por Permutación del Score
3. Explicabilidad Local (Por Transacción): Análisis de Intervención Contrafactual
4. Visualización de Cascada Forense (*Waterfall Attribution Plot*) para un Caso de Fraude
5. Extracción de Reglas para Motores de Reglas Bancarios mediante Árbol Sustituto
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


In [ ]:
# Creación de un Dataset Bancario de Tarjetas de Crédito con 6 Variables Interpretables
rng = np.random.default_rng(42)
n_legit = 1200
df_legit = pd.DataFrame({
    'monto_usd': rng.lognormal(3.8, 0.6, n_legit).clip(5, 600),
    'distancia_domicilio_km': rng.exponential(8.0, n_legit).clip(0.1, 60),
    'intentos_pin_fallidos': rng.choice([0, 1], size=n_legit, p=[0.96, 0.04]),
    'velocidad_compras_1h': rng.poisson(1.2, n_legit).clip(1, 5),
    'ratio_ vs_promedio_cliente': rng.normal(1.0, 0.25, n_legit).clip(0.2, 2.2),
    'antiguedad_comercio_meses': rng.normal(48, 15, n_legit).clip(6, 120)
})

n_fraud = 45
df_fraud = pd.DataFrame({
    'monto_usd': rng.uniform(1800, 6500, n_fraud),
    'distancia_domicilio_km': rng.uniform(450, 3200, n_fraud),
    'intentos_pin_fallidos': rng.choice([2, 3, 4], size=n_fraud, p=[0.3, 0.5, 0.2]),
    'velocidad_compras_1h': rng.integers(7, 18, n_fraud),
    'ratio_ vs_promedio_cliente': rng.uniform(6.5, 18.0, n_fraud),
    'antiguedad_comercio_meses': rng.uniform(0.1, 3.0, n_fraud)
})

df_tx = pd.concat([df_legit, df_fraud], ignore_index=True)
df_tx.index = [f'TX_{i:04d}' for i in range(1, len(df_tx) + 1)]

iforest = IsolationForest(n_estimators=200, contamination=45/1245, random_state=42)
predicciones = iforest.fit_predict(df_tx)
puntajes_anomalia = -iforest.decision_function(df_tx)  # > 0 es anomalia
print(f"Transacciones totales: {len(df_tx)} | Alertas detectadas por Isolation Forest: {(predicciones == -1).sum()}")


## 1. Explicabilidad Local Contrafactual: ¿Por qué se Bloqueó la Transacción `TX_1201`?

Para explicar individualmente una transacción marcada como anomalía $x^*$, aplicamos el principio de **intervención marginal contrafactual**:
1. Calculamos el puntaje de anomalía original $S(x^*)$.
2. Para cada variable $j \in \{1, \dots, p\}$, creamos una copia de la transacción $x^{(-j)}$ donde reemplazamos únicamente el valor de la variable $j$ por la **mediana histórica de las transacciones legítimas**.
3. Medimos cuánto disminuye el puntaje de anomalía al "corregir" esa variable: $\Delta S_j = S(x^*) - S(x^{(-j)})$.

Las variables con mayor $\Delta S_j$ son exactamente las culpables de que los árboles aislaran prematuramente a esa transacción.


In [ ]:
def explicar_transaccion_individual(modelo, df_datos, id_transaccion, medianas_normales):
    x_orig = df_datos.loc[[id_transaccion]].copy()
    score_orig = -modelo.decision_function(x_orig)[0]
    
    impactos = {}
    for col in df_datos.columns:
        x_mod = x_orig.copy()
        x_mod[col] = medianas_normales[col]
        score_mod = -modelo.decision_function(x_mod)[0]
        impactos[col] = max(0.0, score_orig - score_mod)
        
    serie_imp = pd.Series(impactos)
    serie_pct = (serie_imp / serie_imp.sum()) * 100 if serie_imp.sum() > 0 else serie_imp
    return score_orig, x_orig.iloc[0], serie_pct.sort_values(ascending=True)

medianas_sanas = df_tx[predicciones == 1].median()
id_caso = 'TX_1205'  # Una de las transacciones fraudulentas
score_caso, valores_caso, contrib_caso = explicar_transaccion_individual(iforest, df_tx, id_caso, medianas_sanas)

fig, ax = plt.subplots(figsize=(10, 5))
colores = plt.cm.Reds(np.linspace(0.35, 0.9, len(contrib_caso)))
etiquetas_barras = [f"{col} (Valor: {valores_caso[col]:.1f} vs Mediana: {medianas_sanas[col]:.1f})" for col in contrib_caso.index]

ax.barh(etiquetas_barras, contrib_caso.values, color=colores, edgecolor='k')
ax.set_xlabel('Contribución Porcentual al Puntaje de Anomalía (%)', fontsize=11, fontweight='bold')
ax.set_title(f'Auditoría Forense Local para {id_caso} (Score de Anomalía = +{score_caso:.3f})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Destilación de Reglas Globales mediante Árbol de Decisión Sustituto

Entrenamos un `DecisionTreeClassifier` de profundidad 3 para explicar las decisiones de `IsolationForest` en lenguaje humano para el equipo de auditoría.


In [ ]:
y_etiquetas_if = (predicciones == -1).astype(int)  # 1 = Alerta Fraude, 0 = Normal
arbol_sustituto = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol_sustituto.fit(df_tx, y_etiquetas_if)

fidelidad = arbol_sustituto.score(df_tx, y_etiquetas_if)
print(f"Fidelidad del Árbol Sustituto replicando a Isolation Forest: {fidelidad*100:.2f}%\n")
print(export_text(arbol_sustituto, feature_names=list(df_tx.columns), decimals=1))

plt.figure(figsize=(15, 6))
plot_tree(arbol_sustituto, feature_names=list(df_tx.columns), 
          class_names=['Legítima', 'ALERTA FRAUDE'], filled=True, rounded=True, fontsize=9.5)
plt.title(f'Reglas de Motor Antifraude Destiladas de Isolation Forest (Fidelidad = {fidelidad*100:.1f}%)', fontsize=12.5, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Conclusiones

1. **Explicabilidad por Intervención Contrafactual**: Permite justificar ante reguladores o clientes exactamente qué atributos dispararon el bloqueo de una transacción individual.
2. **Árbol Sustituto**: Convierte un bosque aleatorio de 200 árboles de aislamiento en 3 o 4 reglas de negocio auditables con $>99\%$ de concordancia.
